# 🎮 Você é o agente

No exemplo 2, um algoritmo aprendeu a cruzar o lago congelado. Agora **você** é o patinador — mas o jogo muda conforme o que você consegue **enxergar**.

| Modo | O que você vê | Conceito da aula |
|---|---|---|
| 🗺️ **Mapa completo** | o lago inteiro | ambiente **totalmente observável** (slide 32) — e você tem um **modelo** do mundo: sabe onde estão os buracos e planeja antes de agir (*model-based*, slide 36) |
| 🌫️ **Neblina** | só as casas vizinhas; o resto você precisa lembrar | ambiente **parcialmente observável** (slide 32) |
| 🔢 **Olhos de agente** | só um **número** (o estado) e a **recompensa** — exatamente o que um algoritmo de RL recebe | sem mapa e sem modelo: aprender **só por tentativa e erro** (*model-free*, slides 16 e 36) |

A caixa **Gelo escorregadio** liga o ambiente estocástico do slide 41: cada ação vai 1/3 para onde você pediu e 1/3 para cada lado.

### 🏆 Missões

| # | Configuração | Missão |
|---|---|---|
| 1 | Mapa completo · 4×4 · gelo firme | Pegue o presente com o **menor número de passos**. Qual é o mínimo? |
| 2 | Mapa completo · 4×4 · **escorregadio** | Jogue 5 episódios. Quantos presentes? Sua estratégia mudou? |
| 3 | **Neblina** · **8×8** · gelo firme | Pegue o presente sem cair. Em quantos episódios? |
| 4 | **Olhos de agente** · 4×4 · gelo firme | Quantos episódios até o **primeiro** presente? Use o diário. Depois clique em **Revelar o mapa** |
| 5 👑 | **Olhos de agente** · 4×4 · **escorregadio** | O desafio do algoritmo. Compare com o botão 🎲 do agente aleatório |

> Rode as células abaixo (as de código ficam recolhidas) e jogue no tabuleiro da seção 2.

## 1. Preparação (rode as células — elas ficam recolhidas)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle, Ellipse, FancyArrow
from IPython.display import display, clear_output, HTML
import ipywidgets as widgets

In [2]:
# @title Ambiente: o lago congelado (o mesmo do exemplo 2)
ACOES = ['cima', 'baixo', 'esquerda', 'direita']
SETAS = {'cima': '↑', 'baixo': '↓', 'esquerda': '←', 'direita': '→'}
MOV = {'cima': (-1, 0), 'baixo': (1, 0), 'esquerda': (0, -1), 'direita': (0, 1)}
LADOS = {'cima': ['esquerda', 'direita'], 'baixo': ['esquerda', 'direita'],
         'esquerda': ['cima', 'baixo'], 'direita': ['cima', 'baixo']}

# S = início, F = gelo firme, H = buraco, G = presente
MAPA_4x4 = ["SFFF",
            "FHFH",
            "FFFH",
            "HFFG"]


class LagoCongelado:
    """reset() -> estado inicial; step(acao) -> (novo_estado, recompensa, fim, acao_real).
    O estado é o número da casa: 0, 1, 2, ... da esquerda para a direita, de cima para baixo."""

    def __init__(self, mapa=MAPA_4x4, escorregadio=False):
        self.mapa = mapa
        self.n_linhas, self.n_colunas = len(mapa), len(mapa[0])
        self.n_estados = self.n_linhas * self.n_colunas
        self.escorregadio = escorregadio
        self.reset()

    def casa(self, estado):
        return self.mapa[estado // self.n_colunas][estado % self.n_colunas]

    def reset(self):
        self.estado = 0
        return self.estado

    def step(self, acao):
        real = acao
        if self.escorregadio:   # 1/3 vai para onde pediu, 1/3 para cada lado
            real = np.random.choice([acao] + LADOS[acao])
        l, c = divmod(self.estado, self.n_colunas)
        dl, dc = MOV[real]
        l = min(max(l + dl, 0), self.n_linhas - 1)   # na borda, fica no lugar
        c = min(max(c + dc, 0), self.n_colunas - 1)
        self.estado = l * self.n_colunas + c
        tipo = self.casa(self.estado)
        recompensa = 1.0 if tipo == 'G' else 0.0     # só o presente dá recompensa
        fim = tipo in 'HG'                           # buraco ou presente: fim do episódio
        return self.estado, recompensa, fim, real

MAPA_8x8 = ["SFFFFFFF",
            "FFFFFFFF",
            "FFFHFFFF",
            "FFFFFHFF",
            "FFFHFFFF",
            "FHHFFFHF",
            "FHFFHFHF",
            "FFFHFFFG"]
MAPAS = {'4×4': MAPA_4x4, '8×8': MAPA_8x8}
MODOS = ['🗺️ Mapa completo', '🌫️ Neblina', '🔢 Olhos de agente']
experimentos = []   # um registro por configuração jogada (veja a seção 3)

In [3]:
# @title Desenho do lago
def desenhar_presente(ax, x, y, s=1.0):
    ax.add_patch(Rectangle((x - .24 * s, y - .24 * s), .48 * s, .4 * s, fc='#E53935', ec='#8E1B18', zorder=5))
    ax.add_patch(Rectangle((x - .05 * s, y - .24 * s), .1 * s, .4 * s, fc='#FFD54F', zorder=6))
    ax.add_patch(Rectangle((x - .24 * s, y - .06 * s), .48 * s, .08 * s, fc='#FFD54F', zorder=6))
    for dx in (-.09, .09):
        ax.add_patch(Circle((x + dx * s, y + .2 * s), .08 * s, fc='none', ec='#FFD54F', lw=2.5, zorder=6))


def desenhar_patinador(ax, x, y):
    ax.add_patch(Circle((x, y), .26, fc='#FF7043', ec='#4E342E', lw=1.5, zorder=10))
    for dx in (-.09, .09):
        ax.add_patch(Circle((x + dx, y + .06), .05, fc='white', zorder=11))
        ax.add_patch(Circle((x + dx, y + .06), .022, fc='#111', zorder=12))
    ax.add_patch(Rectangle((x - .2, y - .12), .4, .07, fc='#1E88E5', zorder=11))   # cachecol


def desenhar_lago(ax, lago, pos=None, rastro=None, valores=None, politica=None, titulo=None,
                  visiveis=None, lembradas=None, numeros=False):
    """Desenha o lago. visiveis/lembradas: conjuntos de casas para o modo neblina."""
    nl, nc = lago.n_linhas, lago.n_colunas
    xy = lambda s: (s % nc + .5, nl - 1 - s // nc + .5)
    for s in range(lago.n_estados):
        x, y = xy(s)
        tipo = lago.casa(s)
        ax.add_patch(Rectangle((x - .5, y - .5), 1, 1, fc='#DDEFF8', ec='white', lw=2))
        if tipo == 'H':
            ax.add_patch(Ellipse((x, y), .8, .62, fc='#0B3D91', ec='#062457', lw=2, zorder=2))
        elif tipo == 'G':
            desenhar_presente(ax, x, y)
        elif tipo == 'S':
            ax.text(x - .44, y + .32, 'início', fontsize=7, color='#546E7A')
        if numeros:
            ax.text(x + .42, y - .44, str(s), fontsize=7, color='#78909C', ha='right', zorder=3)
        if valores is not None and tipo not in 'HG':
            ax.text(x, y - .38, f'{valores[s]:.2f}', ha='center', fontsize=9, color='#263238', zorder=4)
        if politica is not None and tipo not in 'HG':
            dl, dc = MOV[politica(s)]
            ax.add_patch(FancyArrow(x - dc * .2, y + .08 + dl * .17, dc * .34, -dl * .3, width=.05,
                                    head_width=.2, head_length=.14, color='#E65100', zorder=7))
        if visiveis is not None and s not in visiveis:
            alfa = .55 if (lembradas and s in lembradas) else 1.0
            ax.add_patch(Rectangle((x - .5, y - .5), 1, 1, fc='#90A4AE', ec='#90A4AE', alpha=alfa, zorder=8))
    if rastro:
        pts = [xy(s) for s in rastro]
        ax.plot([p[0] for p in pts], [p[1] for p in pts], '-', color='#FF7043', alpha=.45, lw=4, zorder=9)
    if pos is not None:
        x, y = xy(pos)
        if lago.casa(pos) == 'H':
            ax.text(x, y, '✗', ha='center', va='center', fontsize=26, color='#FF7043', fontweight='bold', zorder=10)
        else:
            desenhar_patinador(ax, x, y)
    ax.set_xlim(0, nc); ax.set_ylim(0, nl); ax.set_aspect('equal'); ax.axis('off')
    if titulo:
        ax.set_title(titulo, fontsize=11, fontweight='bold')

In [ ]:
# @title O jogo (botões, modos e diário)
def jogo():
    w_mapa = widgets.ToggleButtons(options=list(MAPAS), description='Mapa:')
    w_modo = widgets.ToggleButtons(options=MODOS, description='Modo:')
    w_liso = widgets.Checkbox(value=False, description='Gelo escorregadio', indent=False)
    bot = {a: widgets.Button(description=SETAS[a], layout=widgets.Layout(width='52px', height='40px'))
           for a in ACOES}
    b_novo = widgets.Button(description='Novo episódio', button_style='info')
    b_revelar = widgets.Button(description='Revelar o mapa', button_style='warning')
    b_aleat = widgets.Button(description='🎲 Aleatório joga 100', layout=widgets.Layout(width='170px'))
    tela, painel, diario, placar = widgets.Output(), widgets.HTML(), widgets.HTML(), widgets.HTML()
    E = {}

    def vizinhas(s):
        lago = E['lago']
        l, c = divmod(s, lago.n_colunas)
        return {ll * lago.n_colunas + cc for ll, cc in [(l, c), (l - 1, c), (l + 1, c), (l, c - 1), (l, c + 1)]
                if 0 <= ll < lago.n_linhas and 0 <= cc < lago.n_colunas}

    def nova_config(_=None):
        lago = LagoCongelado(MAPAS[w_mapa.value], escorregadio=w_liso.value)
        reg = {'jogador': 'você', 'modo': w_modo.value.split(' ', 1)[1], 'mapa': w_mapa.value,
               'escorregadio': 'sim' if w_liso.value else 'não', 'episodios': 0, 'presentes': 0,
               'passos': 0, '1º presente no episódio': None}
        experimentos.append(reg)
        E.update(lago=lago, reg=reg, vistas=set(), revelar=False, aviso='')
        novo_episodio()

    def novo_episodio(_=None):
        s = E['lago'].reset()
        E.update(s=s, rastro=[s], log=[], fim=False, r=0.0, passos=0, msg='', revelar=False)
        E['vistas'] |= vizinhas(s)
        b_revelar.description = 'Revelar o mapa'
        redesenhar()

    def agir(a):
        if E['fim']:
            E['msg'] = 'O episódio acabou: clique em <b>Novo episódio</b>.'
            redesenhar()
            return
        s = E['s']
        s2, r, fim, real = E['lago'].step(a)
        E['log'].append((E['passos'], s, a, r, s2, fim))
        E['rastro'].append(s2)
        E.update(s=s2, r=r, fim=fim, passos=E['passos'] + 1)
        E['vistas'] |= vizinhas(s2)
        olhos = w_modo.value == MODOS[2]
        E['msg'] = ''
        if real != a and not olhos:
            E['msg'] = f'Escorregou! Você pediu {SETAS[a]} e o gelo te levou para {SETAS[real]}.'
        if fim:
            reg = E['reg']
            reg['episodios'] += 1
            reg['passos'] += E['passos']
            if r > 0:
                reg['presentes'] += 1
                if reg['1º presente no episódio'] is None:
                    reg['1º presente no episódio'] = reg['episodios']
                E['msg'] += f' <b style="color:#2E7D32">🎁 Presente em {E["passos"]} passos! Recompensa +1.</b>'
            elif olhos:
                E['msg'] += ' <b style="color:#C62828">O episódio terminou com recompensa 0.</b> (O que aconteceu? Você não sabe...)'
            else:
                E['msg'] += ' <b style="color:#C62828">Caiu no buraco! Fim do episódio, recompensa 0.</b>'
        redesenhar()

    def revelar(_=None):
        E['revelar'] = not E['revelar']
        b_revelar.description = 'Esconder o mapa' if E['revelar'] else 'Revelar o mapa'
        redesenhar()

    def aleatorio(_=None):
        lago = LagoCongelado(MAPAS[w_mapa.value], escorregadio=w_liso.value)
        presentes = 0
        for _ in range(100):
            s, fim, n = lago.reset(), False, 0
            while not fim and n < 200:
                s, r, fim, _r = lago.step(np.random.choice(ACOES))
                n += 1
            presentes += r > 0
        experimentos.append({'jogador': 'aleatório', 'modo': '—', 'mapa': w_mapa.value,
                             'escorregadio': 'sim' if w_liso.value else 'não', 'episodios': 100,
                             'presentes': int(presentes), 'passos': np.nan, '1º presente no episódio': None})
        E['aviso'] = f'🎲 O agente aleatório jogou 100 episódios neste lago e pegou <b>{presentes}</b> presentes.'
        redesenhar()

    def cartao():
        s, r, fim, n = E['s'], E['r'], E['fim'], E['lago'].n_estados
        if fim:
            status = f'<span style="color:{"#2E7D32" if r > 0 else "#C62828"}">terminou</span>'
        else:
            status = 'continua'
        return (f'<div style="font-family:sans-serif;border:2px solid #455A64;border-radius:14px;padding:18px 24px;'
                f'width:260px;background:#ECEFF1">'
                f'<div style="font-size:12px;color:#607D8B">o que o agente recebe do ambiente</div>'
                f'<div style="font-size:15px;margin-top:10px">estado</div>'
                f'<div style="font-size:72px;font-weight:bold;line-height:1;color:#263238">{s}</div>'
                f'<div style="font-size:13px;color:#607D8B">um entre {n} estados (0 a {n - 1})</div>'
                f'<div style="font-size:16px;margin-top:12px">recompensa: <b>{r:+.0f}</b></div>'
                f'<div style="font-size:16px">episódio: <b>{status}</b></div></div>')

    def redesenhar():
        lago, modo = E['lago'], w_modo.value
        with tela:
            clear_output(wait=True)
            if modo == MODOS[2] and not E['revelar']:
                display(HTML(cartao()))
            else:
                tam = 3.8 if lago.n_linhas == 4 else 5.2
                fig, ax = plt.subplots(figsize=(tam, tam))
                if E['revelar']:
                    desenhar_lago(ax, lago, pos=E['s'], rastro=E['rastro'], numeros=True,
                                  titulo='O mapa revelado (números = estados)')
                elif modo == MODOS[1]:
                    desenhar_lago(ax, lago, pos=E['s'], rastro=E['rastro'],
                                  visiveis=vizinhas(E['s']), lembradas=E['vistas'])
                else:
                    desenhar_lago(ax, lago, pos=E['s'], rastro=E['rastro'])
                plt.show()
                plt.close(fig)
        painel.value = f'<div style="min-height:22px">{E["msg"]}</div>'
        linhas = ''.join(f'<tr><td>{t}</td><td>{s}</td><td>{SETAS[a]}</td><td>{r:+.0f}</td><td>{s2}</td>'
                         f'<td>{"sim" if f else "não"}</td></tr>' for t, s, a, r, s2, f in E['log'][-10:])
        diario.value = ('<b>Diário do episódio</b> (últimos 10 passos)'
                        '<table style="font-family:monospace;border-collapse:collapse" cellpadding="3">'
                        '<tr style="background:#ECEFF1"><th>t</th><th>estado</th><th>ação</th><th>recompensa</th>'
                        f'<th>novo estado</th><th>fim?</th></tr>{linhas}</table>')
        reg = E['reg']
        primeiro = reg['1º presente no episódio'] or '—'
        placar.value = (f'<b>Você nesta configuração:</b> {reg["episodios"]} episódios · '
                        f'{reg["presentes"]} presentes · 1º presente no episódio {primeiro}'
                        f'<br>{E["aviso"]}')

    for a, b in bot.items():
        b.on_click(lambda _, a=a: agir(a))
    b_novo.on_click(novo_episodio)
    b_revelar.on_click(revelar)
    b_aleat.on_click(aleatorio)
    for w in (w_mapa, w_modo, w_liso):
        w.observe(nova_config, names='value')

    vazio = lambda: widgets.Label(' ', layout=widgets.Layout(width='56px'))
    cruz = widgets.VBox([widgets.HBox([vazio(), bot['cima']]),
                         widgets.HBox([bot['esquerda'], vazio(), bot['direita']]),
                         widgets.HBox([vazio(), bot['baixo']])])
    lado = widgets.VBox([cruz, b_novo, b_revelar, b_aleat, painel, diario],
                        layout=widgets.Layout(margin='0 0 0 24px'))
    nova_config()
    display(widgets.VBox([w_modo, widgets.HBox([w_mapa, w_liso]), placar, widgets.HBox([tela, lado])]))

## 2. Jogue

Escolha o modo, o mapa e o gelo — e cumpra as missões da tabela lá em cima. Trocar qualquer opção começa uma **nova configuração** (o placar zera e o resultado anterior fica guardado para a seção 3).

In [ ]:
jogo()

## 3. Seus resultados

Rode a célula abaixo depois de jogar: ela junta todas as configurações que você (e o agente aleatório 🎲) jogaram.

In [4]:
def resumo():
    linhas = [r for r in experimentos if r['episodios'] > 0]
    if not linhas:
        print('Ainda não há episódios: jogue na seção 2 e rode esta célula de novo.')
        return
    df = pd.DataFrame(linhas)
    df['taxa de presentes'] = (df['presentes'] / df['episodios']).map('{:.0%}'.format)
    df['passos por episódio'] = (df['passos'] / df['episodios']).round(1)
    df['1º presente no episódio'] = [('—' if pd.isna(x) else int(x)) for x in df['1º presente no episódio']]
    colunas = ['jogador', 'modo', 'mapa', 'escorregadio', 'episodios', 'presentes', 'taxa de presentes',
               'passos por episódio', '1º presente no episódio']
    display(df[colunas].fillna('—'))


resumo()

Ainda não há episódios: jogue na seção 2 e rode esta célula de novo.


## Para pensar

1. **Missão 1:** com o mapa à vista você planejou o caminho inteiro **antes** de clicar. Isso é usar um **modelo** do mundo (slide 36). Que informação você tinha que um algoritmo de RL normalmente **não** tem?
2. **Missão 2:** com o gelo escorregadio, você passou a evitar casas **ao lado** de buracos, mesmo que o caminho ficasse mais longo? Esse é o trade-off **curto × longo prazo** do slide 29.
3. **Missão 3:** na neblina, você usou a **memória** (as casas acinzentadas). Num ambiente parcialmente observável, o estado atual basta para decidir? O que a propriedade de Markov (slide 30) diria?
4. **Missão 4:** nos olhos de agente, recompensa 0 aparece tanto em "andei para uma casa segura" quanto em "caí no buraco". Como você descobriu o que era o quê? É exatamente o problema que o algoritmo do exemplo 2 resolveu com o caderninho.
5. **Missão 5:** compare seus presentes com os do 🎲 aleatório. Você foi melhor? Quantos episódios **você** precisaria para ficar tão bom quanto o agente treinado do exemplo 2 (≈ 60%)? E ele precisou de quantos?

> **Moral da história:** RL é aprender a agir quando você **só** vê estados e recompensas. Nas próximas aulas, Markov e MDP dão a matemática do "onde estou e para onde posso ir"; Q-Learning transforma o diário em uma tabela de valores.